In [ ]:
import os
import httpx
from dotenv import load_dotenv

load_dotenv()

API_KEY = os.getenv("CMC_API_KEY")

print("API key loaded:", API_KEY is not None)

In [ ]:
url = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/quotes/latest"

headers = {
    "X-CMC_PRO_API_KEY": API_KEY,
    "Accept": "application/json",
}

params = {
    "symbol": "BTC",
    "convert": "USD"
}

response = httpx.get(
    url,
    headers=headers,
    params=params
)

print(response.status_code)


In [ ]:
data = response.json()

data

In [ ]:
data.keys()

In [ ]:
type(data['data'])

In [ ]:
len(data['data'])


In [ ]:
for i in range(13):
    print(data['data'][i]['slug'])

In [ ]:
list(data['data'][0].keys())

In [ ]:
url = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/quotes/latest"

headers = {
    "X-CMC_PRO_API_KEY": API_KEY,
    "Accept": "application/json",
}

params = {
    "id": "1",
    "convert": "USD"
}

response = httpx.get(
    url,
    headers=headers,
    params=params
)


data = response.json()

for i in range(len(data['data'])):
    print(data['data'][i]['slug'])


In [ ]:
url = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/quotes/latest"

headers = {
    "X-CMC_PRO_API_KEY": API_KEY,
    "Accept": "application/json",
}

params = {
    "slug": "bitcoin",
    "convert": "USD"
}

response = httpx.get(
    url,
    headers=headers,
    params=params
)


data = response.json()

for i in range(len(data['data'])):
    print(data['data'][i]['slug'])

In [ ]:
url = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/quotes/historical"

params = {
    "id": "1,1027,5426",
    "time_start": "2026-08-31",
    "time_end": "2026-09-7",
    "interval": "5m",
    "convert": "USD"
}

response = httpx.get(
    url,
    headers=headers,
    params=params
)

data = response.json()

data['data']

In [ ]:
import polars as pl

quotes = {}
ids = [1, 1027, 5426]
asset_names = []
symbols = []

for id in ids:
    quotes[data['data'][str(id)]['name']] = data['data'][str(id)]['quotes']
    asset_names.append(data['data'][str(id)]['name'])
    symbols.append(data['data'][str(id)]['symbol'])


list_dataframe = []

for name, symbol in zip(asset_names, symbols):

    asset_quotes = quotes[name]

    dataframe = pl.DataFrame({
        "timestamp": [q['timestamp'] for q in asset_quotes],
        "asset": [name] * len(asset_quotes),
        "symbol": [symbol] * len(asset_quotes),
        "price": [q["quote"]["USD"]["price"] for q in asset_quotes],
    })

    dataframe = dataframe.with_columns(
        pl.col("timestamp")
        .str.to_datetime(time_zone="UTC")
        .alias("timestamp")
    )

    list_dataframe.append(dataframe)

full_dataframe = pl.concat(list_dataframe)

full_dataframe

In [ ]:
import polars as pl

for name in asset_names:
        dataframe = pl.DataFrame({
            "timestamp": [q[name]['timestamp'] for q in quotes],
            "price": [q[name]["quote"]["USD"]["pricee"] for q in quotes]
        })

In [ ]:
url = "https://pro-api.coinmarketcap.com/v3/cryptocurrency/quotes/historical"

params = {
    "id": "1",
    "time_start": "2026-08-31",
    "time_end": "2026-09-7",
    "interval": "5m",
    "convert": "USD"
}

response = httpx.get(
    url,
    headers=headers,
    params=params
)

data = response.json()


quotes = data["data"]["1"]["quotes"]

len(quotes)


In [ ]:
import polars as pl

btc = pl.DataFrame({
    "timestamp": [q["timestamp"] for q in quotes],
    "price": [q["quote"]["USD"]["price"] for q in quotes],
})



btc

In [ ]:
btc = btc.with_columns(
    pl.col("timestamp")
    .str.to_datetime(time_zone="UTC")
    .alias("timestamp")
)

btc.head()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    btc["timestamp"].to_list(),
    btc["price"].to_list(),
)

ax.set_title("Bitcoin Price — 5 Minute Data")
ax.set_xlabel("Time")
ax.set_ylabel("BTC Price (USD)")

ax.yaxis.set_major_formatter(
    StrMethodFormatter("${x:,.0f}")
)

ax.grid(alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
url = "https://pro-api.coinmarketcap.com/v2/cryptocurrency/ohlcv/historical"

params = {
    "id": "1",
    "time_start": "2026-08-31",
    "time_end": "2026-09-07",
    "time_period": "hourly",
    "interval": "1h",
    "convert": "USD",
}

response = httpx.get(
    url,
    headers=headers,
    params=params,
)

response.raise_for_status()

data = response.json()

In [ ]:
quotes = data["data"]["quotes"]
quotes

In [ ]:
btc = pl.DataFrame({
    "timestamp": [q['time_open'] for q in quotes],
    "open": [q["quote"]["USD"]["open"] for q in quotes],
    "high": [q["quote"]["USD"]["high"] for q in quotes],
    "low": [q["quote"]["USD"]["low"] for q in quotes],
    "close": [q["quote"]["USD"]["close"] for q in quotes],
    "volume": [q["quote"]["USD"]["volume"] for q in quotes],
})

btc

In [ ]:
btc = btc.with_columns(
    pl.col("timestamp")
    .str.to_datetime(time_zone="UTC")
    .alias("timestamp")
)

btc.head()

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import StrMethodFormatter

fig, ax = plt.subplots(figsize=(14, 6))

ax.plot(
    btc["timestamp"].to_list(),
    btc["close"].to_list(),
)

ax.set_title("Bitcoin Price —  Hourly Data")
ax.set_xlabel("Time")
ax.set_ylabel("BTC Price (USD)")

ax.yaxis.set_major_formatter(
    StrMethodFormatter("${x:,.0f}")
)

ax.grid(alpha=0.3)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
import httpx
import os
from dotenv import load_dotenv
import polars as pl

load_dotenv()

API_KEY = os.getenv("CMC_API_KEY")

headers = {
    "X-CMC_PRO_API_KEY": API_KEY,
    "Accept": "application/json",
}

url = (
    "https://pro-api.coinmarketcap.com"
    "/v5/cryptocurrency/derivatives/market-pairs/list/latest"
)

params = {
    "crypto_id": 1,
    "category": "perpetual",
    "limit": 20,
    "convert": "USD",
}

response = httpx.get(
    url,
    headers=headers,
    params=params,
)

print(response.status_code)

data = response.json()
data

In [ ]:
type(data)

In [ ]:
list(data['data']['market_pairs'][0].keys())



In [ ]:
data['data']['market_pairs'][0]['exchange_reported_quotes'][0]

In [ ]:
data['data']['market_pairs'][0]['quotes'][0]

In [ ]:
market_pairs = data["data"]["market_pairs"]

rows = []

for market in market_pairs:

    reported = market["exchange_reported_quotes"][0]

    rows.append({
        "timestamp": reported["last_updated"],
        "crypto_id": data["data"]["crypto_id"],
        "asset": data["data"]["crypto_name"],
        "symbol": data["data"]["symbol"],

        "market_id": market["market_id"],
        "market_pair": market["market_pair_symbol"],
        "category": market["category"],

        "exchange_id": market["exchange"]["exchange_id"],
        "exchange": market["exchange"]["exchange_name"],

        "quote_symbol": market["market_pair_quote"]["symbol"],

        "price": reported["price"],
        "volume_24h_base": reported["volume_24h_base"],
        "volume_24h_quote": reported["volume_24h_quote"],
        "open_interest": reported["open_interest"],
        "index_price": reported["index_price"],
        "index_basis": reported["index_basis"],
        "funding_rate": reported["funding_rate"],

        "outlier_detected": market["outlier_detected"],
        "exclusions": market["exclusions"],
    })

df = pl.DataFrame(rows)

In [ ]:
df.head()

In [ ]:
pl.Config.set_tbl_rows(-1)
pl.Config.set_tbl_cols(-1)

In [ ]:
print(df.select([
    "exchange",
    "open_interest",
    "funding_rate",
    "index_basis",
    "outlier_detected",
]))

In [ ]:
print(df.select([
    "exchange",
    "market_pair",
    "quote_symbol",
    "open_interest",
    "funding_rate",
    "index_basis",
    "outlier_detected",
    "exclusions",
]).sort(
    "open_interest",
    descending=True
))

In [ ]:
url = (
    "https://pro-api.coinmarketcap.com"
    "/v5/derivatives/liquidations/cryptocurrency/list/latest"
)

params = {
    "crypto_id": "1,1027,5426",
    "convert": "USD",
}

response = httpx.get(
    url,
    headers=headers,
    params=params,
)

print(response.status_code)

liquidation_data = response.json()

liquidation_data

In [ ]:
liquidation_data["data"].keys()

In [ ]:
liquidation_data["data"]['cryptocurrencies'][0].keys()

In [ ]:
list(liquidation_data["data"]['cryptocurrencies'][0]['quotes'][0].keys())

In [ ]:
btc = liquidation_data["data"]["cryptocurrencies"][0]
q = btc["quotes"][0]

imbalance_1h = (
    q["long_liquidations_1h"]
    - q["short_liquidations_1h"]
) / q["total_liquidations_1h"]

imbalance_1h

In [ ]:
cryptocurrencies = liquidation_data["data"]["cryptocurrencies"]

rows = []

for crypto in cryptocurrencies:

    quote = crypto["quotes"][0]

    rows.append({
        "timestamp": quote["last_updated"],
        "crypto_id": crypto["crypto_id"],
        "asset": crypto["name"],
        "symbol": crypto["symbol"],

        "total_liquidations_1h": quote["total_liquidations_1h"],
        "long_liquidations_1h": quote["long_liquidations_1h"],
        "short_liquidations_1h": quote["short_liquidations_1h"],

        "total_liquidations_4h": quote["total_liquidations_4h"],
        "long_liquidations_4h": quote["long_liquidations_4h"],
        "short_liquidations_4h": quote["short_liquidations_4h"],

        "total_liquidations_24h": quote["total_liquidations_24h"],
        "long_liquidations_24h": quote["long_liquidations_24h"],
        "short_liquidations_24h": quote["short_liquidations_24h"],
    })

liq_df = pl.DataFrame(rows)

liq_df = liq_df.with_columns(
    pl.col("timestamp")
    .str.to_datetime(time_zone="UTC")
)

liq_df

In [ ]:
liq_df = liq_df.with_columns(

    (
        (
            pl.col("long_liquidations_1h")
            - pl.col("short_liquidations_1h")
        )
        / pl.col("total_liquidations_1h")
    ).alias("liquidation_imbalance_1h"),

    (
        pl.col("long_liquidations_1h")
        / pl.col("total_liquidations_1h")
    ).alias("long_liquidation_share_1h"),

    (
        pl.col("total_liquidations_1h")
        / (pl.col("total_liquidations_24h") / 24)
    ).alias("liquidation_acceleration_1h_vs_24h"),
)

In [ ]:
liq_df

In [ ]:
liq_df = liq_df.with_columns(

    (
        pl.col("total_liquidations_1h")
        /
        (
            (
                pl.col("total_liquidations_4h")
                - pl.col("total_liquidations_1h")
            ) / 3
        )
    ).alias("liquidation_intensity_1h_vs_prev3h"),

    (
        pl.col("total_liquidations_1h")
        /
        (pl.col("total_liquidations_24h") / 24)
    ).alias("liquidation_intensity_1h_vs_24h"),

)

In [ ]:
liq_df = liq_df.with_columns(

    (
        (
            pl.col("long_liquidations_1h")
            - pl.col("short_liquidations_1h")
        )
        / pl.col("total_liquidations_1h")
    ).alias("liquidation_imbalance_1h"),

    (
        (
            pl.col("long_liquidations_4h")
            - pl.col("short_liquidations_4h")
        )
        / pl.col("total_liquidations_4h")
    ).alias("liquidation_imbalance_4h"),

    (
        (
            pl.col("long_liquidations_24h")
            - pl.col("short_liquidations_24h")
        )
        / pl.col("total_liquidations_24h")
    ).alias("liquidation_imbalance_24h"),

)

In [ ]:
url = (
    "https://pro-api.coinmarketcap.com"
    "/v5/derivatives/liquidations/quotes/latest"
)

params = {
    "convert": "USD",
}

response = httpx.get(
    url,
    headers=headers,
    params=params,
)

print(response.status_code)

global_liquidation_data = response.json()

global_liquidation_data

In [ ]:
url = (
    "https://pro-api.coinmarketcap.com"
    "/v5/derivatives/liquidations/exchange/list/latest"
)

params = {
    "start": 1,
    "limit": 50,
    "convert": "USD",
}

response = httpx.get(
    url,
    headers=headers,
    params=params,
)

print(response.status_code)

exchange_liquidation_data = response.json()

exchange_liquidation_data

In [ ]:
exchange_liquidation_data['data']['exchanges'][0].keys()

In [ ]:
exchange_liquidation_data['data']['exchanges'][0]['quotes']

In [ ]:
quote = global_liquidation_data["data"]["quotes"][0]

global_liq_df = pl.DataFrame({
    "timestamp": [quote["last_updated"]],

    "total_liquidations_1h": [quote["total_liquidations_1h"]],
    "long_liquidations_1h": [quote["long_liquidations_1h"]],
    "short_liquidations_1h": [quote["short_liquidations_1h"]],

    "total_liquidations_4h": [quote["total_liquidations_4h"]],
    "long_liquidations_4h": [quote["long_liquidations_4h"]],
    "short_liquidations_4h": [quote["short_liquidations_4h"]],

    "total_liquidations_24h": [quote["total_liquidations_24h"]],
    "long_liquidations_24h": [quote["long_liquidations_24h"]],
    "short_liquidations_24h": [quote["short_liquidations_24h"]],
})

global_liq_df = global_liq_df.with_columns(
    pl.col("timestamp")
    .str.to_datetime(time_zone="UTC")
)

In [ ]:
quote = global_liquidation_data["data"]["quotes"][0]

global_liq_df = pl.DataFrame({
    "timestamp": [quote["last_updated"]],

    "total_liquidations_1h": [quote["total_liquidations_1h"]],
    "long_liquidations_1h": [quote["long_liquidations_1h"]],
    "short_liquidations_1h": [quote["short_liquidations_1h"]],

    "total_liquidations_4h": [quote["total_liquidations_4h"]],
    "long_liquidations_4h": [quote["long_liquidations_4h"]],
    "short_liquidations_4h": [quote["short_liquidations_4h"]],

    "total_liquidations_24h": [quote["total_liquidations_24h"]],
    "long_liquidations_24h": [quote["long_liquidations_24h"]],
    "short_liquidations_24h": [quote["short_liquidations_24h"]],
})

global_liq_df = global_liq_df.with_columns(
    pl.col("timestamp")
    .str.to_datetime(time_zone="UTC")
)

In [ ]:
exchanges = exchange_liquidation_data["data"]["exchanges"]

rows = []

for exchange in exchanges:

    quote = exchange["quotes"][0]

    rows.append({
        "timestamp": quote["last_updated"],

        "exchange_id": exchange["exchange_id"],
        "exchange": exchange["name"],
        "slug": exchange["slug"],

        "total_liquidations_1h": quote["total_liquidations_1h"],
        "long_liquidations_1h": quote["long_liquidations_1h"],
        "short_liquidations_1h": quote["short_liquidations_1h"],

        "total_liquidations_4h": quote["total_liquidations_4h"],
        "long_liquidations_4h": quote["long_liquidations_4h"],
        "short_liquidations_4h": quote["short_liquidations_4h"],

        "total_liquidations_24h": quote["total_liquidations_24h"],
        "long_liquidations_24h": quote["long_liquidations_24h"],
        "short_liquidations_24h": quote["short_liquidations_24h"],
    })

exchange_liq_df = pl.DataFrame(rows)

exchange_liq_df = exchange_liq_df.with_columns(
    pl.col("timestamp")
    .str.to_datetime(time_zone="UTC")
)

In [ ]:
exchange_liq_df = exchange_liq_df.with_columns(

    (
        (
            pl.col("long_liquidations_1h")
            - pl.col("short_liquidations_1h")
        )
        / pl.col("total_liquidations_1h")
    ).alias("liquidation_imbalance_1h"),

    (
        (
            pl.col("long_liquidations_4h")
            - pl.col("short_liquidations_4h")
        )
        / pl.col("total_liquidations_4h")
    ).alias("liquidation_imbalance_4h"),

    (
        (
            pl.col("long_liquidations_24h")
            - pl.col("short_liquidations_24h")
        )
        / pl.col("total_liquidations_24h")
    ).alias("liquidation_imbalance_24h"),

    (
        pl.col("total_liquidations_1h")
        / (pl.col("total_liquidations_24h") / 24)
    ).alias("liquidation_intensity_1h_vs_24h"),

    (
        pl.col("total_liquidations_1h")
        /
        (
            (
                pl.col("total_liquidations_4h")
                - pl.col("total_liquidations_1h")
            ) / 3
        )
    ).alias("liquidation_intensity_1h_vs_prev3h"),
)

In [ ]:
exchange_liq_df = exchange_liq_df.with_columns(

    (
        pl.col("total_liquidations_1h")
        / pl.col("total_liquidations_1h").sum()
    ).alias("liquidation_share_1h"),

    (
        pl.col("total_liquidations_4h")
        / pl.col("total_liquidations_4h").sum()
    ).alias("liquidation_share_4h"),

    (
        pl.col("total_liquidations_24h")
        / pl.col("total_liquidations_24h").sum()
    ).alias("liquidation_share_24h"),
)

In [ ]:
exchange_liq_df.select([
    "exchange",
    "total_liquidations_1h",
    "liquidation_share_1h",
    "liquidation_imbalance_1h",
    "liquidation_intensity_1h_vs_24h",
]).sort(
    "total_liquidations_1h",
    descending=True,
)

In [ ]:
exchange_total_1h = exchange_liq_df["total_liquidations_1h"].sum()

global_total_1h = global_liq_df["total_liquidations_1h"][0]

print("Global 1h:", global_total_1h)
print("Exchange sum 1h:", exchange_total_1h)
print("Difference:", global_total_1h - exchange_total_1h)

In [ ]:
for horizon in ["1h", "4h", "24h"]:

    exchange_total = exchange_liq_df[
        f"total_liquidations_{horizon}"
    ].sum()

    global_total = global_liq_df[
        f"total_liquidations_{horizon}"
    ][0]

    print(
        horizon,
        "global:",
        global_total,
        "exchange sum:",
        exchange_total,
        "difference:",
        global_total - exchange_total,
    )

In [ ]:
global_1h = global_liq_df["total_liquidations_1h"][0]

exchange_liq_df = exchange_liq_df.with_columns(
    (
        pl.col("total_liquidations_1h")
        / global_1h
    ).alias("global_liquidation_share_1h")
)

In [ ]:
unattributed_1h = (
    global_liq_df["total_liquidations_1h"][0]
    - exchange_liq_df["total_liquidations_1h"].sum()
)